In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
url = "https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main"

resp = requests.get(url)

print(resp.status_code)

200


In [3]:
zf = zipfile.ZipFile(io.BytesIO(resp.content))

len(zf.infolist())

1490

In [4]:
repository_data = []

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    if not filename.endswith(".md"):
        continue

    with zf.open(file_info) as f_in:
        content = f_in.read().decode("utf-8", errors="ignore")

        post = frontmatter.loads(content)

        data = post.to_dict()

        data["filename"] = filename

        repository_data.append(data)

In [5]:
len(repository_data)

1216

In [6]:
repository_data[0]

{'description': 'Review and process open FAQ PRs',
 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep -

In [7]:
def read_repo_data(repo_owner, repo_name):

    prefix = "https://codeload.github.com"

    url = f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"

    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception("Download failed")

    repository_data = []

    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():

        filename = file_info.filename
        filename_lower = filename.lower()

        if not (
            filename_lower.endswith(".md")
            or filename_lower.endswith(".mdx")
        ):
            continue

        try:
            with zf.open(file_info) as f_in:

                content = f_in.read().decode(
                    "utf-8",
                    errors="ignore"
                )

                post = frontmatter.loads(content)

                data = post.to_dict()

                data["filename"] = filename

                repository_data.append(data)

        except Exception as e:
            print(e)

    zf.close()

    return repository_data

In [8]:
faq_data = read_repo_data(
    "DataTalksClub",
    "faq"
)

len(faq_data)

1216

In [9]:
my_repo = read_repo_data(
    "ezhilarasi0509",
    "AI-Agent-"
)

len(my_repo)

2

In [10]:
evidently_docs = read_repo_data(
    "evidentlyai",
    "docs"
)

len(evidently_docs)

95

In [11]:
evidently_docs[45]

{'title': 'LLM regression testing',
 'description': 'How to run regression testing for LLM outputs.',
 'content': 'In this tutorial, you will learn how to perform regression testing for LLM outputs.\n\nYou can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.\n\n<Info>\n  **This example uses Evidently Cloud.** You\'ll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>\n\n# Tutorial scope\n\nHere\'s what we\'ll do:\n\n* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.\n\n* **Get new answers**. Imitate generating new answers to the same question.\n\n* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to eval

In [12]:
len(evidently_docs[45]["content"])

21712

In [13]:
def sliding_window(seq, size, step):

    if size <= 0 or step <= 0:
        raise ValueError(
            "size and step must be positive"
        )

    n = len(seq)

    result = []

    for i in range(0, n, step):

        chunk = seq[i:i + size]

        result.append({
            "start": i,
            "chunk": chunk
        })

        if i + size >= n:
            break

    return result

In [14]:
text = evidently_docs[45]["content"]

chunks = sliding_window(
    text,
    2000,
    1000
)

len(chunks)

21

In [15]:
chunks[0]

{'start': 0,
 'chunk': "In this tutorial, you will learn how to perform regression testing for LLM outputs.\n\nYou can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.\n\n<Info>\n  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>\n\n# Tutorial scope\n\nHere's what we'll do:\n\n* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.\n\n* **Get new answers**. Imitate generating new answers to the same question.\n\n* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to evaluate length, correctness and style consistency.\n\n* **Build a monitoring Dashboard**. Get plo

In [16]:
evidently_chunks = []

for doc in evidently_docs:

    doc_copy = doc.copy()

    doc_content = doc_copy.pop("content")

    chunks = sliding_window(
        doc_content,
        2000,
        1000
    )

    for chunk in chunks:

        chunk.update(doc_copy)

    evidently_chunks.extend(chunks)

In [17]:
len(evidently_chunks)

573

In [18]:
evidently_chunks[0]

{'start': 0,
 'chunk': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>\n\n## Authentication\n\nAll API endpoints are authenticated using Bearer tokens and picked up from the specification file.\n\n```json\n"security": [\n  {\n    "bearerAuth": []\n  }\n]\n```',
 'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'docs-main/api-reference/introduction.mdx'}

In [19]:
import json

with open(
    "evidently_chunks.json",
    "w"
) as f:

    json.dump(
        evidently_chunks,
        f,
        indent=2
    )

In [20]:
import re

text = evidently_docs[45]["content"]

paragraphs = re.split(
    r"\n\s*\n",
    text.strip()
)

len(paragraphs)

153

In [21]:
paragraphs[0]

'In this tutorial, you will learn how to perform regression testing for LLM outputs.'

In [22]:
import re

def split_markdown_by_level(
    text,
    level=2
):

    header_pattern = (
        r'^(#{'
        + str(level)
        + r'} )(.+)$'
    )

    pattern = re.compile(
        header_pattern,
        re.MULTILINE
    )

    parts = pattern.split(text)

    sections = []

    for i in range(1, len(parts), 3):

        header = (
            parts[i]
            + parts[i + 1]
        )

        header = header.strip()

        content = ""

        if i + 2 < len(parts):

            content = parts[i + 2].strip()

        if content:

            section = (
                f"{header}\n\n{content}"
            )

        else:
            section = header

        sections.append(section)

    return sections

In [23]:
sections = split_markdown_by_level(
    text,
    level=2
)

len(sections)

8

In [24]:
sections[0]

'## 1. Installation and Imports\n\nInstall Evidently:\n\n```python\npip install evidently[llm] \n```\n\nImport the required modules:\n\n```python\nimport pandas as pd\nfrom evidently.future.datasets import Dataset\nfrom evidently.future.datasets import DataDefinition\nfrom evidently.future.datasets import Descriptor\nfrom evidently.future.descriptors import *\nfrom evidently.future.report import Report\nfrom evidently.future.presets import TextEvals\nfrom evidently.future.metrics import *\nfrom evidently.future.tests import *\n\nfrom evidently.features.llm_judge import BinaryClassificationPromptTemplate\n```\n\nTo connect to Evidently Cloud:\n\n```python\nfrom evidently.ui.workspace.cloud import CloudWorkspace\n```\n\n**Optional.** To create monitoring panels as code:\n\n```python\nfrom evidently.ui.dashboards import DashboardPanelPlot\nfrom evidently.ui.dashboards import DashboardPanelTestSuite\nfrom evidently.ui.dashboards import DashboardPanelTestSuiteCounter\nfrom evidently.ui.dash

In [25]:
section_chunks = []

for doc in evidently_docs:

    doc_copy = doc.copy()

    doc_content = doc_copy.pop(
        "content"
    )

    sections = split_markdown_by_level(
        doc_content,
        level=2
    )

    for section in sections:

        section_doc = doc_copy.copy()

        section_doc["section"] = section

        section_chunks.append(
            section_doc
        )

In [26]:
len(section_chunks)

264

In [27]:
section_chunks[0]

{'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'docs-main/api-reference/introduction.mdx',
 'section': '## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>'}

In [28]:
with open(
    "section_chunks.json",
    "w"
) as f:

    json.dump(
        section_chunks,
        f,
        indent=2
    )

In [29]:
evidently_docs = read_repo_data("evidentlyai", "docs")

evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop("content")

    chunks = sliding_window(doc_content, 2000, 1000)

    for chunk in chunks:
        chunk.update(doc_copy)

    evidently_chunks.extend(chunks)

len(evidently_chunks)

573

In [30]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(evidently_chunks)

In [31]:
query = "What should be in a test dataset for AI evaluation?"

results = index.search(query, num_results=5)

results

[{'start': 0,
  'chunk': 'Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.\n\nInstead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.\n\n## Create a RAG test dataset\n\nYou can generate ground truth RAG dataset from your data source.\n\n### 1. Create a Project\n\nIn the Evidently UI, start a new Project or open an existing one.\n\n* Navigate to “Datasets” in the left menu.\n* Click “Generate” and select the “RAG” option.\n\n![](/images/synthetic/synthetic_data_select_method.png)\n\n### 2. Upload your knowledge base\n\nSelect a file containing the information your AI system retrieves from. Supported formats: Markdown (.md), CSV, TXT, PDFs. Choose how many inputs to generate.\n\n![](/images/synthetic/synthetic_data_inputs_exa

In [32]:
evidently_chunks[0]

{'start': 0,
 'chunk': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>\n\n## Authentication\n\nAll API endpoints are authenticated using Bearer tokens and picked up from the specification file.\n\n```json\n"security": [\n  {\n    "bearerAuth": []\n  }\n]\n```',
 'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'docs-main/api-reference/introduction.mdx'}

In [33]:
from minsearch import Index

text_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

text_index.fit(evidently_chunks)

In [34]:
query = "What should be in a test dataset for AI evaluation?"

text_results = text_index.search(query, num_results=5)

text_results

[{'start': 0,
  'chunk': 'Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.\n\nInstead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.\n\n## Create a RAG test dataset\n\nYou can generate ground truth RAG dataset from your data source.\n\n### 1. Create a Project\n\nIn the Evidently UI, start a new Project or open an existing one.\n\n* Navigate to “Datasets” in the left menu.\n* Click “Generate” and select the “RAG” option.\n\n![](/images/synthetic/synthetic_data_select_method.png)\n\n### 2. Upload your knowledge base\n\nSelect a file containing the information your AI system retrieves from. Supported formats: Markdown (.md), CSV, TXT, PDFs. Choose how many inputs to generate.\n\n![](/images/synthetic/synthetic_data_inputs_exa

In [35]:
for r in text_results:
    print("FILENAME:", r["filename"])
    print("TITLE:", r.get("title"))
    print("CHUNK:", r["chunk"][:500])
    print("-" * 80)

FILENAME: docs-main/synthetic-data/rag_data.mdx
TITLE: RAG evaluation dataset
CHUNK: Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.

Instead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.

## Create a RAG test dataset

You can generate ground truth RAG dataset from your data source.

### 1. Create a Project

In th
--------------------------------------------------------------------------------
FILENAME: docs-main/quickstart_llm.mdx
TITLE: LLM Evaluation
CHUNK: s:

```python
eval_dataset.as_dataframe()
```

![](/images/examples/llm_quickstart_preview.png)

<Info>
  **What other evals are there?** Browse all available descriptors including deterministic checks, semantic similarity, and LLM judges in the [descriptor list](/

In [36]:
from sentence_transformers import SentenceTransformer
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [37]:
evidently_embeddings = []

for d in tqdm(evidently_chunks):
    v = embedding_model.encode(d["chunk"])
    evidently_embeddings.append(v)

evidently_embeddings = np.array(evidently_embeddings)

evidently_embeddings.shape

  0%|          | 0/573 [00:00<?, ?it/s]

(573, 768)

In [38]:
vector_index = VectorSearch()

vector_index.fit(evidently_embeddings, evidently_chunks)

In [39]:
query = "What should be in a test dataset for AI evaluation?"

q = embedding_model.encode(query)

vector_results = vector_index.search(q, num_results=5)

vector_results

[{'start': 0,
  'chunk': "When working on an AI system, you need test data to run automated evaluations for quality and safety. A test dataset is a structured set of test cases. It can contain:\n\n* Just the inputs, or\n* Both inputs and expected outputs (ground truth).\n\nYou can use this test dataset to:\n\n* Run **experiments** and track if changes improve or degrade system performance.\n* Run **regression testing** to ensure updates don’t break what was already working.\n* **Stress-test** your system with complex or adversarial inputs to check its resilience.\n\n![](/images/synthetic/synthetic_experiments_img.png)\n\nYou can create test datasets manually, collect them from real or historical data, or generate them synthetically. While real data is best, it is not always available or sufficient to cover all cases. Public LLM benchmarks help with general model comparisons but don’t reflect your specific use case. Manually writing test cases takes time and effort.\n\n**Synthetic data 

In [40]:
for r in vector_results:
    print("FILENAME:", r["filename"])
    print("TITLE:", r.get("title"))
    print("CHUNK:", r["chunk"][:500])
    print("-" * 80)

FILENAME: docs-main/synthetic-data/why_synthetic.mdx
TITLE: Why synthetic data?
CHUNK: When working on an AI system, you need test data to run automated evaluations for quality and safety. A test dataset is a structured set of test cases. It can contain:

* Just the inputs, or
* Both inputs and expected outputs (ground truth).

You can use this test dataset to:

* Run **experiments** and track if changes improve or degrade system performance.
* Run **regression testing** to ensure updates don’t break what was already working.
* **Stress-test** your system with complex or adversari
--------------------------------------------------------------------------------
FILENAME: docs-main/synthetic-data/why_synthetic.mdx
TITLE: Why synthetic data?
CHUNK:  you are:

* You're starting from scratch and don’t have real data.
* You need to scale a manually designed dataset with more variation.
* You want to test edge cases, adversarial inputs, or system robustness.
* You're evaluating complex AI sys

In [41]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    q = embedding_model.encode(query)
    return vector_index.search(q, num_results=num_results)


def hybrid_search(query, num_results=5):
    text_results = text_search(query, num_results)
    vector_results = vector_search(query, num_results)

    seen = set()
    final_results = []

    for result in text_results + vector_results:
        key = result.get("filename", "") + result.get("chunk", "")[:100]

        if key not in seen:
            seen.add(key)
            final_results.append(result)

    return final_results

In [42]:
query = "How can I evaluate model performance?"

hybrid_results = hybrid_search(query, num_results=5)

for r in hybrid_results:
    print("FILENAME:", r["filename"])
    print("TITLE:", r.get("title"))
    print("CHUNK:", r["chunk"][:500])
    print("-" * 80)

FILENAME: docs-main/examples/LLM_judge.mdx
TITLE: LLM as a judge
CHUNK: import CloudSignup from '/snippets/cloud_signup.mdx';
import CreateProject from '/snippets/create_project.mdx';

In this tutorial, we'll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.

<Info>
  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.
</Info>

We'll explore two ways to use an LL
--------------------------------------------------------------------------------
FILENAME: docs-main/examples/LLM_judge.mdx
TITLE: LLM as a judge
CHUNK: view the scored dataset in Python. This will show a DataFrame with newly added scores and explanations.

```python
eval_dataset.as_dataframe()
```

![](/images/examples/llm_judge_tutorial_judge_scored_data-min.png)

<Info>
  **Note**: your explanations will vary since LLMs 

In [43]:
query = "How can I evaluate model performance?"

print("TEXT SEARCH RESULTS")
print("=" * 80)

for r in text_search(query, 3):
    print(r["filename"])
    print(r["chunk"][:300])
    print("-" * 80)


print("VECTOR SEARCH RESULTS")
print("=" * 80)

for r in vector_search(query, 3):
    print(r["filename"])
    print(r["chunk"][:300])
    print("-" * 80)


print("HYBRID SEARCH RESULTS")
print("=" * 80)

for r in hybrid_search(query, 3):
    print(r["filename"])
    print(r["chunk"][:300])
    print("-" * 80)

TEXT SEARCH RESULTS
docs-main/examples/LLM_judge.mdx
import CloudSignup from '/snippets/cloud_signup.mdx';
import CreateProject from '/snippets/create_project.mdx';

In this tutorial, we'll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.

<Info>
  **This is a local example.** You will run and ex
--------------------------------------------------------------------------------
docs-main/examples/LLM_judge.mdx
view the scored dataset in Python. This will show a DataFrame with newly added scores and explanations.

```python
eval_dataset.as_dataframe()
```

![](/images/examples/llm_judge_tutorial_judge_scored_data-min.png)

<Info>
  **Note**: your explanations will vary since LLMs are non-deterministic.
</I
--------------------------------------------------------------------------------
docs-main/examples/LLM_judge.mdx
s/customize_llm_judge#change-the-evaluator-llm) to see how you can select a different evaluator LLM. 
</Info>

## 2. 

In [44]:
# Day 4: Agents and Tools
hybrid_search("How can I evaluate model performance?")

[{'start': 0,
  'chunk': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different contex

In [45]:
from typing import List, Any
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

In [46]:
def search_docs(query: str) -> List[Any]:
    """
    Search the documentation chunks using hybrid search.
    """
    return hybrid_search(query, num_results=5)

In [47]:
system_prompt = """
You are a helpful AI documentation assistant.

Always use the search_docs tool before answering.
Answer based on retrieved documentation whenever possible.
If the retrieved documentation is not enough, clearly say that you could not find enough information.
Keep answers clear, practical, and beginner-friendly.
"""

In [48]:
ollama_model = OpenAIChatModel(
    model_name="mistral",
    provider=OpenAIProvider(
        base_url="http://localhost:11434/v1",
        api_key="ollama"
    )
)

In [49]:
agent = Agent(
    model=ollama_model,
    name="docs_agent",
    instructions=system_prompt,
    tools=[search_docs]
)

In [50]:
question = "How can I evaluate model performance?"

result = await agent.run(user_prompt=question)

print(result.output)

 To evaluate model performance in your application built using Hugging Face's Transformers library, you can follow these common evaluation metrics and steps:

1. Split the dataset into train, validation, and test sets. It is essential to have well-balanced datasets if dealing with imbalanced data issues.

```python
from torch.utils.data import random_split
train_size = len(dataset) * 0.7
val_size = len(dataset) * 0.15
test_size = len(dataset) * 0.15

train_data, val_data, test_data = random_split(dataset, [train_size, val_size, test_size])
```

2. Implement and use evaluation metrics according to your specific task (e.g., classification, sequence-to-sequence, and others).
Here we provide an example using the common ClassificationMetrics class for evaluation in classification tasks:

```python
from transformers import EvalPrediction, ClassificationReport

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    labels = np.

In [51]:
questions = [
    "How can I evaluate model performance?",
    "What should be in a test dataset for AI evaluation?",
    "What is Evidently used for?"
]

for q in questions:
    result = await agent.run(user_prompt=q)
    print("QUESTION:", q)
    print("ANSWER:", result.output)
    print("-" * 80)

QUESTION: How can I evaluate model performance?
ANSWER:  To evaluate model performance, you can use several metrics depending on the type of problem your model is solving. Here are some common evaluation metrics:

1. Accuracy (classification problems): The percentage of correct predictions out of total predictions. `accuracy = (TP + TN) / (TP + TN + FP + FN)`
   - True Positives (TP): Correctly predicted positive examples
   - True Negatives (TN): Correctly predicted negative examples
   - False Positives (FP): Incorrectly predicted positive examples when the actual label is negative
   - False Negatives (FN): Incorrectly predicted negative examples when the actual label is positive

2. Precision (classification problems): The ratio of correctly predicted positives to all predictions made. `precision = TP / (TP + FP)`
3. Recall (classification problems): The ratio of correctly predicted positives to the total number of actual positive examples. `recall = TP / (TP + FN)`
4. F1-score: Th

In [52]:
# Day 5: Offline Evaluation and Testing
question = "How can I evaluate model performance?"

result = await agent.run(user_prompt=question)

print(result.output)

 To evaluate a machine learning model's performance, you can utilize various evaluation metrics based on the specific problem you are solving and the type of output your model generates. Here are some common examples:

1. Classification problems:
   - **Accuracy**: a percentage indicating how many correct predictions out of total predictions the model made.
   - **Precision**: a measurement of the model's ability to correctly identify true positives (positive instances that the model predicted as such).
   - **Recall (Sensitivity)**: a measurement of the model's ability to find all positive instances in the data, even if some are misclassified as negatives.
   - **F1-Score**: a measure combining precision and recall; it takes the harmonic mean of both, providing an overall score that balances precision and recall.

2. Regression problems:
   - **Mean Absolute Error (MAE)**: calculates the average magnitude of errors between observed and predicted values.
   - **Root Mean Squared Error 

In [53]:
import json
import secrets
from pathlib import Path
from datetime import datetime

In [54]:
LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

def save_interaction_log(question, answer, source="user", model_name="mistral"):
    """
    Save one agent interaction as a JSON log file.
    """

    log_data = {
        "timestamp": datetime.now().isoformat(),
        "agent_name": "docs_agent_ollama",
        "model": model_name,
        "source": source,
        "system_prompt": system_prompt,
        "question": question,
        "answer": answer,
        "tool_used": "search_docs",
    }

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"docs_agent_ollama_{ts}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f:
        json.dump(log_data, f, indent=2)

    return filepath

In [55]:
question = "How can I evaluate model performance?"

result = await agent.run(user_prompt=question)

answer = result.output

print(answer)

log_file = save_interaction_log(question, answer, source="user", model_name="mistral")

print("Saved log:", log_file)

 To evaluate model performance in my context, you can follow these steps:

1. Split your data into training and testing sets. This allows you to train the model on a portion of the data, and then test its accuracy on the remaining portion.

```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

2. Train the model using the training data.

```python
model = MyModel()  # Replace this with your trained model
model.fit(X_train, y_train)
```

3. Evaluate the performance of the model on the testing set by invoking the model's evaluation methods. Common metrics are accuracy, precision, recall, F1 score, and AUC-ROC (AUC-Relevance-Ordinal Curve).

```python
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(

In [56]:
list(LOG_DIR.glob("*.json"))

[PosixPath('logs/docs_agent_ollama_20260527_081149_0924af.json')]

In [57]:
test_questions = [
    "How can I evaluate model performance?",
    "What should be in a test dataset for AI evaluation?",
    "How can I compare model performance?",
    "What is Evidently used for?",
    "How do I monitor data drift?",
    "What is a report in Evidently?",
    "How can I create a dashboard?",
    "How do I test model quality?",
    "What are metrics in Evidently?",
    "How can I evaluate a machine learning model?"
]

In [58]:
for q in test_questions:
    print("QUESTION:", q)

    result = await agent.run(user_prompt=q)
    answer = result.output

    print("ANSWER:", answer[:500])
    print("-" * 80)

    save_interaction_log(
        question=q,
        answer=answer,
        source="manual-test",
        model_name="mistral"
    )

QUESTION: How can I evaluate model performance?
ANSWER:  To evaluate the performance of a machine learning model, there are several common metrics to consider:

1. Accuracy: This metric measures how often the model makes correct predictions among all predictions. However, it might not be useful for imbalanced datasets where one class is much more frequent than another.

2. Precision: Measures the proportion of true positive predictions among all positive predictions. It's particularly important when you have a low number of false positives compared t
--------------------------------------------------------------------------------
QUESTION: What should be in a test dataset for AI evaluation?
ANSWER:  To create a test dataset for an AI model's evaluation, it's generally recommended to follow these guidelines:

1. **Diversity**: The data should be diverse and representative of the real-world scenarios your model will encounter. This could include various inputs from different sources with

In [59]:
logs = list(LOG_DIR.glob("*.json"))
len(logs)

11

In [60]:
def evaluate_answer(log_record):
    """
    Simple offline evaluation for one logged answer.
    """

    question = log_record["question"]
    answer = log_record["answer"]

    checks = {
        "answer_not_empty": len(answer.strip()) > 20,
        "answer_relevant": any(
            word.lower() in answer.lower()
            for word in question.split()
            if len(word) > 4
        ),
        "answer_clear": len(answer.split()) >= 20,
        "tool_used": log_record.get("tool_used") == "search_docs",
        "has_contextual_answer": not any(
            phrase in answer.lower()
            for phrase in [
                "i don't know",
                "i do not know",
                "cannot answer",
                "no information"
            ]
        )
    }

    score = sum(checks.values()) / len(checks)

    return {
        "question": question,
        "answer": answer,
        **checks,
        "score": score
    }

In [61]:
def load_log_file(log_file):
    with open(log_file, "r", encoding="utf-8") as f:
        return json.load(f)


eval_rows = []

for log_file in LOG_DIR.glob("*.json"):
    log_record = load_log_file(log_file)
    eval_result = evaluate_answer(log_record)
    eval_result["file"] = log_file.name
    eval_rows.append(eval_result)

len(eval_rows)

11

In [62]:
import pandas as pd

df_evals = pd.DataFrame(eval_rows)

df_evals.head()

,question,answer,answer_not_empty,answer_relevant,answer_clear,tool_used,has_contextual_answer,score,file
0,What are metrics in Evidently?,"According to the Evidently documentation, met...",True,True,True,True,True,1.0,docs_agent_ollama_20260527_081519_710bdc.json
1,How can I evaluate a machine learning model?,"To evaluate a machine learning model, there a...",True,True,True,True,True,1.0,docs_agent_ollama_20260527_081548_717b79.json
2,How can I evaluate model performance?,To evaluate the performance of a machine lear...,True,True,True,True,True,1.0,docs_agent_ollama_20260527_081301_d32f5f.json
3,How can I create a dashboard?,"To create a dashboard using this service, fol...",True,True,True,True,True,1.0,docs_agent_ollama_20260527_081453_c93c32.json
4,What is Evidently used for?,Evidently is a data quality tool that helps u...,True,True,True,True,True,1.0,docs_agent_ollama_20260527_081339_a6bf06.json


In [63]:
df_evals[[
    "answer_not_empty",
    "answer_relevant",
    "answer_clear",
    "tool_used",
    "has_contextual_answer",
    "score"
]].mean(numeric_only=True)

answer_not_empty         1.0
answer_relevant          1.0
answer_clear             1.0
tool_used                1.0
has_contextual_answer    1.0
score                    1.0
dtype: float64

In [64]:
df_evals.to_csv("day5_evaluation_report.csv", index=False)

print("Saved: day5_evaluation_report.csv")

Saved: day5_evaluation_report.csv
